In [36]:
import json
import string
import random
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')     
nltk.download('wordnet')   


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [37]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [38]:
data_f =  open('data.json').read()
data = json.loads(data_f)

In [39]:
lemmatizer = WordNetLemmatizer()

all_words = []
classes = []
doc_x = []
doc_y = []
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

for obj in data["ourIntents"]:
    for patterns in obj['patterns']:
        tokens = nltk.word_tokenize(patterns.lower())
        clean_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in string.punctuation and word not in stop_words]
        all_words.extend(clean_tokens)
        doc_x.append(patterns)
        doc_y.append(obj['tag'])

    if obj['tag'] not in classes:
        classes.append(obj['tag'])

all_words = sorted(set(all_words))
classes = sorted(set(classes))

In [40]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(doc_y)

training_data = []

for index, doc in enumerate(doc_x):
    bow =  [1 if word in doc else 0 for word in all_words]
    training_data.append((bow, encoded_labels[index]))

import random
random.shuffle(training_data)

x=[]
y=[]

for feature, label in training_data:
    x.append(feature)
    y.append(label)

x= np.array(x)
y=np.array(y)

In [41]:
random.shuffle(training_data)

training_data = np.array(training_data, dtype = object)

xtrain = np.array(list(training_data[:,0]))
ytrain = np.array(list(training_data[:,1]))

In [42]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

ytrain = to_categorical(ytrain)
model = Sequential()
model.add(Dense(128, input_shape =(len(xtrain[0]),), activation = 'relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation = 'relu'))
model.add(Dropout(0.3))
model.add(Dense(len(ytrain[0]), activation = 'softmax'))

C:\Users\Hp\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [43]:
optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01)
model.compile(loss='categorical_crossentropy', optimizer = optimizer, metrics = ['accuracy'])

history = model.fit(xtrain, ytrain, epochs =160, verbose=1)

Epoch 1/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0369 - loss: 3.2159
Epoch 2/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2269 - loss: 3.0966 
Epoch 3/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3251 - loss: 2.9042 
Epoch 4/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2619 - loss: 2.6950 
Epoch 5/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3624 - loss: 2.5238 
Epoch 6/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3513 - loss: 2.2937 
Epoch 7/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4247 - loss: 2.2087 
Epoch 8/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5474 - loss: 1.8838 
Epoch 9/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5359 - loss: 1.6975 
Epoch 10/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5676 - loss: 1.5421 
Epoch 11/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6205 - loss: 1.4413 
Epoch 12/160
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6364 - los

In [44]:
import pickle

model.save("chatbot_m.h5")

with open("tokenizer_data.pkl","wb") as f:
    pickle.dump({'all_words': all_words, 'label_encoder': label_encoder}, f)

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

tfidf_vectorizer = TfidfVectorizer()
x_tfidf = tfidf_vectorizer.fit_transform(doc_x)
svm_model = CalibratedClassifierCV(SVC(kernel='linear'))
svm_model.fit(x_tfidf, encoded_labels)

C:\Users\Hp\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


CalibratedClassifierCV(estimator=SVC(kernel='linear'))

In [54]:
from tensorflow.keras.models import load_model

model = load_model("chatbot_m.h5")

with open("tokenizer_data.pkl", "rb") as f:
    data = pickle.load(f)

all_words = data['all_words']
label_encoder = data['label_encoder']

with open("data.json") as f:
    intents = json.load(f)

def clean_text(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in string.punctuation and word not in stop_words]
    return tokens
    

def bag_words(text, all_words):
    tokens = clean_text(text)
    bow = np.array([1 if word in tokens else 0 for word in all_words])
    return bow

def predict_intent(text):
    bow_vector = bag_words(text, all_words).reshape(1,-1)
    prediction = model.predict(bow_vector, verbose = 0)
    predicted_class = np.argmax(prediction)
    intent = label_encoder.inverse_transform([predicted_class])[0]
    return intent, np.max(prediction)

def predict_tfidf_intent(text):
    ctext = clean_text(text)
    cstext = "".join(ctext)
    tfidf_input = tfidf_vectorizer.transform([cstext])
    prediction_probs = svm_model.predict_proba(tfidf_input)[0]
    predicted_index = np.argmax(prediction_probs)
    predicted_label = label_encoder.inverse_transform([predicted_index])[0]
    confidence = prediction_probs[predicted_index]
    return predicted_label, confidence

def get_response(intent_name, intents_json):
    for intent in intents_json["ourIntents"]:
        if intent['tag'] == intent_name:
            return random.choice(intent["responses"])
    return "I'm not sure how to respond to that"

while True:
    user_input= input("You : ")
    user_input = user_input.lower()
    intent, confidence = predict_intent(user_input)
    intent_tfidf, confidence_tfidf = predict_tfidf_intent(user_input)
    print(f"(Debug) Predicted intent bow+NN: {intent} with confidence: {confidence:.2f}")
    print(f"(Debug) Predicted intent TF-IDF[SVM]: {intent_tfidf} with confidence : {confidence_tfidf:.2f}")

    if confidence>0.3:
        final_intent = intent
    elif confidence_tfidf > confidence:
        final_intent = intent_tfidf
    else:
        final_intent = None

    if final_intent:
         response = get_response(final_intent, intents)
    else:
        response = "I'm not sure how to respond to that."

    print("Bot :", response)
    if intent =='goodbye':
        break

You :  hi


(Debug) Predicted intent bow+NN: greeting with confidence: 0.54
(Debug) Predicted intent TF-IDF[SVM]: greeting with confidence : 0.10
Bot : Hi there


You :  hi


(Debug) Predicted intent bow+NN: greeting with confidence: 0.54
(Debug) Predicted intent TF-IDF[SVM]: greeting with confidence : 0.10
Bot : Hi there


You :  hello


...

(Debug) Predicted intent bow+NN: jokes with confidence: 1.00
(Debug) Predicted intent TF-IDF[SVM]: jokes with confidence : 0.22
Bot : I ate a clock yesterday, it was very time-consuming


You :  joke


(Debug) Predicted intent bow+NN: jokes with confidence: 1.00
(Debug) Predicted intent TF-IDF[SVM]: jokes with confidence : 0.22
Bot : As I get older and I remember all the people Iâ€™ve lost along the way, I think to myself, maybe a career as a tour guide wasnâ€™t for me.


You :  end


(Debug) Predicted intent bow+NN: goodbye with confidence: 1.00
(Debug) Predicted intent TF-IDF[SVM]: goodbye with confidence : 0.10
Bot : Goodbye! Have a great day!
